$\textbf{ Using Qwen2.5-0.5B-Instruct }$

$\textbf{ 0) Imports}$

In [1]:

import math, gc, os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)


/home/maxmagnusson/project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


$\textbf{ 1) Dataset handling}$

In [2]:
class RedditGuidelineDataset(Dataset):
    def __init__(self, df):
        self.df = df.fillna("").reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            "body": r.get("body",""),
            "rule": r.get("rule",""),
            "subreddit": r.get("subreddit",""),
            "pos1": r.get("positive_example_1",""),
            "pos2": r.get("positive_example_2",""),
            "neg1": r.get("negative_example_1",""),
            "neg2": r.get("negative_example_2",""),
            "label": int(r["rule_violation"]),
        }


$\textbf{ 2) Collate}$

In [3]:
def collate_batch(batch, tok, cap_len=768, body_cap=384, rule_cap=128, pad_to_multiple_of=8):
    PAD = tok.pad_token_id
    BOS = getattr(tok, "bos_token_id", None)
    EOS = tok.eos_token_id

    hdr_comment = tok("### Comment\n", add_special_tokens=False)["input_ids"]
    hdr_rule    = tok("### Rule\n", add_special_tokens=False)["input_ids"]
    hdr_ctx     = tok("### Context\n", add_special_tokens=False)["input_ids"]
    sep_ids     = tok("\n\n", add_special_tokens=False)["input_ids"]

    def s(x):
        if x is None: return ""
        if isinstance(x, float):
            try:
                if np.isnan(x): return ""
            except Exception: pass
        return str(x)

    def enc(ex):
        body_ids = tok(s(ex["body"]), truncation=True, max_length=body_cap, add_special_tokens=False)["input_ids"]
        rule_ids = tok("Rule: " + s(ex["rule"]), truncation=True, max_length=rule_cap, add_special_tokens=False)["input_ids"]
        extras = "\n".join([
            "Subreddit: " + s(ex["subreddit"]),
            "Positive: " + s(ex["pos1"]) + " || " + s(ex["pos2"]),
            "Negative: " + s(ex["neg1"]) + " || " + s(ex["neg2"]),
        ])
        static = len(hdr_comment)+len(sep_ids)+len(hdr_rule)+len(sep_ids)+len(hdr_ctx)
        overhead = (1 if BOS is not None else 0) + (1 if EOS is not None else 0)
        remaining = max(0, cap_len - overhead - static - len(body_ids) - len(rule_ids))
        extra_ids = tok(extras, truncation=True, max_length=remaining, add_special_tokens=False)["input_ids"]

        ids = []
        if BOS is not None: ids.append(BOS)
        ids += hdr_comment + body_ids + sep_ids + hdr_rule + rule_ids + sep_ids + hdr_ctx + extra_ids
        if EOS is not None: ids.append(EOS)
        return ids

    ids_list = [enc(x) for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)

    maxlen = max(len(x) for x in ids_list)
    if pad_to_multiple_of: maxlen += (-maxlen) % pad_to_multiple_of

    input_ids = torch.full((len(ids_list), maxlen), PAD, dtype=torch.long)
    attention_mask = torch.zeros((len(ids_list), maxlen), dtype=torch.bool)
    for i, ids in enumerate(ids_list):
        L = min(len(ids), maxlen)
        input_ids[i, :L] = torch.tensor(ids[:L], dtype=torch.long)
        attention_mask[i, :L] = True

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

$\textbf{ 3) Load and build DataLoaders}$

In [ ]:
name = "Models/qwen3-transformer" 

train_df = pd.read_csv("./DataFolder/train_masked.csv")
val_df   = pd.read_csv("./DataFolder/val_masked.csv")

label_counts = train_df["rule_violation"].value_counts().to_dict()
weights = train_df["rule_violation"].map(lambda y: 1.0 / label_counts[y]).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

tok = AutoTokenizer.from_pretrained(name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

train_loader = DataLoader(
    RedditGuidelineDataset(train_df),
    batch_size=16,
    sampler=sampler,  
    collate_fn=lambda b: collate_batch(b, tok, cap_len=768, body_cap=384, rule_cap=160),
    pin_memory=True, num_workers=4, persistent_workers=True,
)

val_loader = DataLoader(
    RedditGuidelineDataset(val_df),
    batch_size=16, shuffle=False,
    collate_fn=lambda b: collate_batch(b, tok, cap_len=768, body_cap=384, rule_cap=160),
    pin_memory=True, num_workers=4, persistent_workers=True,
)


$\textbf{ 4) Model with proper classification}$

In [5]:
device = "cuda"

model = AutoModelForSequenceClassification.from_pretrained(
    name,
    num_labels=2,
    problem_type="single_label_classification",
)
model.config.pad_token_id = tok.pad_token_id
model.config.label_smoothing_factor = 0.1

if hasattr(model.config, "hidden_dropout_prob"):
    model.config.hidden_dropout_prob = 0.1
if hasattr(model.config, "attention_probs_dropout_prob"):
    model.config.attention_probs_dropout_prob = 0.1


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Models/qwen2.5-instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


$\textbf{5: Unfreeze layers}$

In [6]:
def unfreeze_qwen2_cls(model, k_last_layers):
    for p in model.parameters():
        p.requires_grad = False

    if not hasattr(model, "model") or not hasattr(model.model, "layers"):
        raise AttributeError("Expected model.model.layers for Qwen2; not found.")
    for p in model.model.layers[-k_last_layers:].parameters():
        p.requires_grad = True

    head_names = ["score", "classifier"]
    found_head = False
    for name_ in head_names:
        if hasattr(model, name_):
            for p in getattr(model, name_).parameters():
                p.requires_grad = True
            found_head = True
            break
    if not found_head:
        raise AttributeError("Could not find classifier head (tried: score, classifier).")

    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"Trainable tensors: {len(trainable)}")
    for n in trainable[:20]:
        print("  ", n)
    if len(trainable) == 0:
        raise RuntimeError("No trainable parameters after unfreeze!")

unfreeze_qwen2_cls(model, k_last_layers=1)


Trainable tensors: 13
   model.layers.23.self_attn.q_proj.weight
   model.layers.23.self_attn.q_proj.bias
   model.layers.23.self_attn.k_proj.weight
   model.layers.23.self_attn.k_proj.bias
   model.layers.23.self_attn.v_proj.weight
   model.layers.23.self_attn.v_proj.bias
   model.layers.23.self_attn.o_proj.weight
   model.layers.23.mlp.gate_proj.weight
   model.layers.23.mlp.up_proj.weight
   model.layers.23.mlp.down_proj.weight
   model.layers.23.input_layernorm.weight
   model.layers.23.post_attention_layernorm.weight
   score.weight


$\textbf{6: Model settings}$

In [ ]:
from torch.cuda.amp import GradScaler
import torch.nn as nn

epochs         = 20                
learning_rate  = 2e-4              
weight_decay   = 0.05
grad_accum     = 4
max_grad_norm  = 1.0

use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

if use_cuda:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

trainable_params = [p for p in model.parameters() if p.requires_grad]

decay_params, nodecay_params = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if n.endswith("bias") or "LayerNorm.weight" in n or "layernorm.weight" in n:
        nodecay_params.append(p)
    else:
        decay_params.append(p)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params,   "weight_decay": weight_decay},
        {"params": nodecay_params, "weight_decay": 0.0},
    ],
    lr=learning_rate,
    betas=(0.9, 0.95),
)

steps_per_epoch    = math.ceil(len(train_loader) / max(1, grad_accum))
num_training_steps = epochs * steps_per_epoch
num_warmup_steps   = int(0.06 * num_training_steps)  # 6% warmup is fine

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = GradScaler(enabled=(torch.cuda.is_available() and amp_dtype == torch.float16))

model.to(device)


/tmp/ipykernel_6415/995060814.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(torch.cuda.is_available() and amp_dtype == torch.float16))


Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rota

$\textbf{7: EMA weights}$

In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
            if getattr(v, "dtype", None) is not None and v.dtype.is_floating_point
        }
        self.backup = {}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def store(self, model):
        self.backup = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
            if k in self.shadow
        }

    @torch.no_grad()
    def copy_to(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                v.copy_(self.shadow[k])

class DummyEMA:
    def update(self, *args, **kwargs): pass
    def store(self, *args, **kwargs): pass
    def copy_to(self, *args, **kwargs): pass

$\textbf{8: Training loop}\\ \\
\text{Early stop}$

In [ ]:
best_val = float("inf")
patience = 2
stuck = 0
best_state = None
ema_enabled = True
ema = EMA(model, decay=0.95) if ema_enabled else DummyEMA()

for epoch in range(1, epochs + 1):
    model.train()
    torch.cuda.empty_cache(); gc.collect()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=amp_dtype):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss / grad_accum

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1
                ema.update(model)
        else:
            loss.backward()
            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                optimizer_steps += 1
                ema.update(model)

        running_loss += loss.detach().item()

    avg_train_loss = running_loss / max(1, len(train_loader))
    print(f"Epoch {epoch} | steps: {optimizer_steps}/{steps_per_epoch} | train loss: {avg_train_loss:.4f}")

    model.eval()
    ema.store(model)     
    ema.copy_to(model)

    correct = 0
    total   = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=amp_dtype):
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss_sum += out.loss.item()

            preds = out.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.numel()

    val_acc  = correct / max(1, total)
    val_loss = val_loss_sum / max(1, len(val_loader))
    print(f"Epoch {epoch}: val loss = {val_loss:.4f} | val acc = {val_acc:.4f}")

    
    if val_loss < best_val:
        best_val = val_loss
        stuck = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stuck += 1
        if stuck > patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"Loaded best checkpoint with val loss {best_val:.4f}")


Epoch 1 | steps: 19/19 | train loss: 0.4606
Epoch 1: val loss = 1.3928 | val acc = 0.4915
Epoch 2 | steps: 19/19 | train loss: 0.2350
Epoch 2: val loss = 0.7255 | val acc = 0.5763
Epoch 3 | steps: 19/19 | train loss: 0.1964
Epoch 3: val loss = 0.7478 | val acc = 0.6068
Epoch 4 | steps: 19/19 | train loss: 0.1746
Epoch 4: val loss = 0.6394 | val acc = 0.6305
Epoch 5 | steps: 19/19 | train loss: 0.1654
Epoch 5: val loss = 0.6513 | val acc = 0.6373
Epoch 6 | steps: 19/19 | train loss: 0.1428
Epoch 6: val loss = 0.5948 | val acc = 0.6610
Epoch 7 | steps: 19/19 | train loss: 0.1365
Epoch 7: val loss = 0.6189 | val acc = 0.6814
Epoch 8 | steps: 19/19 | train loss: 0.1222
Epoch 8: val loss = 0.6154 | val acc = 0.6576
Epoch 9 | steps: 19/19 | train loss: 0.1172
Epoch 9: val loss = 0.6151 | val acc = 0.6542
Early stopping.
Loaded best checkpoint with val loss 0.5948
